In [1]:
# ⚙️ Imports and Config
import os, json, uuid, fitz, pytesseract, faiss
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Any
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)

EMBED_MODEL = "paraphrase-multilingual-mpnet-base-v2"
FAISS_DIR = "./vector_store"
os.makedirs(FAISS_DIR, exist_ok=True)
TESSERACT_LANG = "kan"
MIN_SCORE = 0.55


d:\MANIPAL_MSIS\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 🧩 Data Structure
@dataclass
class Doc:
    page_content: str
    metadata: Dict[str, Any]


In [3]:
# 🧾 PDF Text Extraction (with OCR fallback)
def extract_text_from_pdf(pdf_path: str) -> List[Doc]:
    docs = []
    pdf = fitz.open(pdf_path)
    for i, page in enumerate(pdf):
        text = page.get_text("text").strip()
        used_ocr = False
        if len(text) < 50:
            from PIL import Image
            import io
            pix = page.get_pixmap(dpi=200)
            image = Image.open(io.BytesIO(pix.tobytes("png")))
            ocr_text = pytesseract.image_to_string(image, lang=TESSERACT_LANG).strip()
            if len(ocr_text) > len(text):
                text, used_ocr = ocr_text, True
        docs.append(Doc(text, {"source_file": os.path.basename(pdf_path), "page": i + 1, "ocr": used_ocr}))
    pdf.close()
    return docs


def process_all_pdfs(folder: str) -> List[Doc]:
    pdfs = list(Path(folder).glob("*.pdf"))
    all_docs = []
    for p in pdfs:
        print(f"📘 Processing {p.name}")
        all_docs.extend(extract_text_from_pdf(str(p)))
    print(f"✅ Loaded {len(all_docs)} pages total.")
    return all_docs


In [4]:
# ✂️ Text Splitting
def split_docs(docs: List[Doc], size=1000, overlap=200):
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    lc_docs = [Document(page_content=d.page_content, metadata=d.metadata) for d in docs]
    out = splitter.split_documents(lc_docs)
    print(f"Split {len(docs)} pages into {len(out)} chunks.")
    return [Doc(o.page_content, o.metadata) for o in out]


In [5]:
# 🔡 Embedding Manager
class EmbeddingManager:
    def __init__(self, model_name=EMBED_MODEL):
        print(f"Loading model {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.dim = self.model.get_sentence_embedding_dimension()
    def embed(self, texts: List[str]):
        return self.model.encode(texts, show_progress_bar=True, convert_to_numpy=True).astype(np.float32)


In [6]:
# 📦 FAISS Vector Store
class FaissVectorStore:
    def __init__(self, dim):
        self.index = faiss.IndexFlatIP(dim)
        self.ids, self.meta = [], []
    def add(self, docs: List[Doc], embs: np.ndarray):
        norms = np.linalg.norm(embs, axis=1, keepdims=True)
        normalized = embs / np.clip(norms, 1e-10, None)
        self.index.add(normalized)
        self.ids.extend([f"id_{uuid.uuid4().hex[:8]}" for _ in docs])
        self.meta.extend([d.metadata for d in docs])
        faiss.write_index(self.index, os.path.join(FAISS_DIR, "faiss.index"))
        np.save(os.path.join(FAISS_DIR, "embeddings.npy"), normalized)
        with open(os.path.join(FAISS_DIR, "meta.json"), "w", encoding="utf-8") as f:
            json.dump(self.meta, f, ensure_ascii=False, indent=2)
        print(f"✅ Added {len(docs)} vectors.")
    def search(self, q_emb, top_k=3):
        q = q_emb.reshape(1, -1)
        q /= np.linalg.norm(q) + 1e-10
        D, I = self.index.search(q.astype(np.float32), top_k)
        return [{"score": float(s), "meta": self.meta[i], "index": int(i)} for s, i in zip(D[0], I[0]) if i < len(self.meta)]


In [7]:
# 🔍 Retriever
class Retriever:
    def __init__(self, emb_mgr, vs): self.emb_mgr, self.vs = emb_mgr, vs
    def retrieve(self, q, top_k=3):
        emb = self.emb_mgr.embed([q])[0]
        res = self.vs.search(emb, top_k)
        return [r for r in res if r["score"] >= MIN_SCORE]


In [8]:
from dotenv import load_dotenv
load_dotenv()


True

In [9]:
# 🧠 LLM Integration (Groq)
groq_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=groq_key, model_name="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)

def groq_invoke(prompt: str) -> str:
    try:
        resp = llm.invoke([prompt])
        return resp.content.strip()
    except Exception as e:
        return f"⚠️ LLM Error: {e}"


In [10]:
# 🧩 Build Index Function
def build_index(folder: str):
    docs = process_all_pdfs(folder)
    chunks = split_docs(docs)
    emb_mgr = EmbeddingManager()
    embs = emb_mgr.embed([c.page_content for c in chunks])
    vs = FaissVectorStore(emb_mgr.dim)
    vs.add(chunks, embs)
    retriever = Retriever(emb_mgr, vs)
    return retriever, chunks


In [12]:
# 🚀 Build / Rebuild Index (run once)
retriever, chunks = build_index(DATA_DIR)


📘 Processing HISTORICAL.pdf
✅ Loaded 60 pages total.
Split 60 pages into 99 chunks.
Loading model paraphrase-multilingual-mpnet-base-v2...


Batches: 100%|██████████| 4/4 [00:08<00:00,  2.23s/it]

✅ Added 99 vectors.


In [13]:
import re

def is_kannada_text(text: str) -> bool:
    """Detect if text is predominantly Kannada (Unicode range U+0C80–U+0CFF)."""
    kannada_chars = re.findall(r'[\u0C80-\u0CFF]', text)
    ratio = len(kannada_chars) / max(len(text), 1)
    return ratio > 0.5  # at least 50% Kannada characters


def rag_simple(query, retriever, chunks, llm_invoke, top_k=3):
    # 1️⃣ Language check
    if not is_kannada_text(query):
        return {
            "status": "invalid_language",
            "answer": "ಯಾವುದೇ ಸಂಬಂಧಿತ ಮಾಹಿತಿ ಕಂಡುಬಂದಿಲ್ಲ",
            "sources": []
        }

    # 2️⃣ Retrieve context
    results = retriever.retrieve(query, top_k=top_k)
    if not results:
        return {
            "status": "no_context",
            "answer": "⚠️ ಯಾವುದೇ ಸಂಬಂಧಿತ ಮಾಹಿತಿ ಕಂಡುಬಂದಿಲ್ಲ.",
            "sources": []
        }

    # 3️⃣ Context threshold (if all below similarity threshold)
    top_sim = max(r["score"] for r in results)
    if top_sim < 0.55:
        return {
            "status": "no_context",
            "answer": "⚠️ ಯಾವುದೇ ಸಂಬಂಧಿತ ಮಾಹಿತಿ ಕಂಡುಬಂದಿಲ್ಲ.",
            "sources": []
        }

    # 4️⃣ Build context
    context = "\n\n".join([chunks[r["index"]].page_content for r in results])
    prompt = f"""
    ಕೆಳಗಿನ ಪಾರ್ಶ್ವಭೂಮಿಯನ್ನು ಉಪಯೋಗಿಸಿ ಪ್ರಶ್ನೆಗೆ ಸರಳವಾಗಿ ಮತ್ತು ಸಂಕ್ಷಿಪ್ತವಾಗಿ ಕನ್ನಡದಲ್ಲಿ ಉತ್ತರಿಸಿ.
    ಉತ್ತರವು 2–3 ವಾಕ್ಯಗಳಲ್ಲಿ ಮಾತ್ರ ಇರಬೇಕು. ಪಾರ್ಶ್ವಭೂಮಿಯಿಂದ ಹೊರಗಿನ ಮಾಹಿತಿ ಸೇರಿಸಬೇಡಿ.

    ಪಾರ್ಶ್ವಭೂಮಿ:
    {context}

    ಪ್ರಶ್ನೆ: {query}

    ಉತ್ತರ (ಕನ್ನಡದಲ್ಲಿ):
    """
    # 5️⃣ LLM answer
    try:
        response_text = llm_invoke(prompt)
    except Exception as e:
        response_text = f"⚠️ LLM Error: {e}"

    return {
        "status": "ok",
        "answer": response_text,
        "sources": [r["meta"] for r in results]
    }


In [18]:
query = "ಹಂಪಿಯ ಇತಿಹಾಸವೇನು?"
answer = rag_simple(query, retriever, chunks, groq_invoke)
print(answer["answer"])


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.71it/s]


ಹಂಪಿಯ ಇತಿಹಾಸ ಭಾರತದ ಇತಿಹಾಸದಲ್ಲಿ ಒಂದು ಮಹತ್ವದ ಅಧ್ಯಾಯವಾಗಿದೆ. ಹಂಪಿ ವಿಜಯನಗರ ಸಾಮ್ರಾಜ್ಯದ ರಾಜಧಾನಿಯಾಗಿತ್ತು, ಇದು 14ನೇ ಶತಮಾನದಲ್ಲಿ ಭಾರತದ ಅತ್ಯಂತ ಶಕ್ತಿಶಾಲಿ ಸಾಮ್ರಾಜ್ಯವಾಗಿತ್ತು. ಹಂಪಿಯ ಇತಿಹಾಸವು ವಿಜಯನಗರ ಸಾಮ್ರಾಜ್ಯದ ಉದಯ, ವಿಸ್ತರಣ ಮತ್ತು ವಿಘಟನೆಯನ್ನು ಒಳಗೊಂಡಿದೆ.


In [15]:
query = "about chitra durga"
answer = rag_simple(query, retriever, chunks, groq_invoke)
print(answer["answer"])


ಯಾವುದೇ ಸಂಬಂಧಿತ ಮಾಹಿತಿ ಕಂಡುಬಂದಿಲ್ಲ


In [ ]:
query = "ಧಾರವಾಡದ ಸಾಂಸ್ಕೃತಿಕ ಮಹತ್ವವೇನು?"
answer = rag_simple(query, retriever, chunks, groq_invoke)
print("🧭 ಪ್ರಶ್ನೆ:", query)
print("💬 ಉತ್ತರ:\n", answer["answer"])

Batches: 100%|██████████| 1/1 [00:00<00:00, 13.88it/s]


🧭 ಪ್ರಶ್ನೆ: ಧಾರವಾಡದ ಸಾಂಸ್ಕೃತಿಕ ಮಹತ್ವವೇನು?
💬 ಉತ್ತರ:
 ಧಾರವಾಡದ ಸಾಂಸ್ಕೃತಿಕ ಮಹತ್ವವು ಚಾಲುಕ್ಯರ ವಿಜ್ಯೇತ್ಸವಗಳ ತಾಣವಾಗಿದೆ. ಇಲ್ಲಿ ದಾರವಿಡ ಮತ್ತು ರ್ನಗರ ಶೈಲಿಗಳ ಸಂಯೋಜನೆಯನ್ನು ಕಾಣಬಹುದು. ವಿರೂಪಾಕ್ಷ ದೇವಾಲಯವು ಲೇಕಮಹಾದೇವಿ ರಾಣಿಯಿಂದ ನಿರ್ಮಿಸಲ್ಪಟ್ಟಿದೆ ಮತ್ತು ಇದು ಕಂಚಿಪುರದ ಕೈಲಾಸರ್ನಥ ದೇವಸ್ಥಾನದ ಮಾದರಿಯಲ್ಲಿದೆ.


In [17]:
# 🧭 Ask Kannada Question
query = "ಅಮೆರಿಕಾದಲ್ಲಿ ಮಾತನಾಡುವ ಮುಖ್ಯ ಭಾಷೆಗಳು ಯಾವುವು?"
answer = rag_simple(query, retriever, chunks, groq_invoke)
print("🧭 ಪ್ರಶ್ನೆ:", query)
print("💬 ಉತ್ತರ:\n", answer["answer"])

Batches: 100%|██████████| 1/1 [00:00<00:00, 12.16it/s]

🧭 ಪ್ರಶ್ನೆ: ಅಮೆರಿಕಾದಲ್ಲಿ ಮಾತನಾಡುವ ಮುಖ್ಯ ಭಾಷೆಗಳು ಯಾವುವು?
💬 ಉತ್ತರ:
 ⚠️ ಯಾವುದೇ ಸಂಬಂಧಿತ ಮಾಹಿತಿ ಕಂಡುಬಂದಿಲ್ಲ.
